 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# RAG Limitations and Use Cases

## ۱. ایده‌ی اصلی

RAG برای «پیدا کن و توضیح بده» عالیه، ولی جواب همه‌ی مسائل نیست. باید بدونی **کِی RAG، کِی SQL، کِی Fine-tuning و کِی یک Prompt ساده** درسته.

## ۲. محدودیت‌های اصلی

| # | محدودیت | مثال HR | راه مقابله |
|---|---|---|---|
| ۱ | وابستگی به کیفیت Retrieval | سؤال با کلمات متفاوت از متن سند | Hybrid، Rerank، Query Rewriting |
| ۲ | از دست رفتن context در chunking | شرط در chunk بعدی افتاده | Overlap، Parent-Child |
| ۳ | سؤال تجمیعی (Aggregation) | «چند نفر مرخصی بدون حقوق گرفتن؟» | SQL / Tool |
| ۴ | سؤال سراسری (Global) | «خلاصه‌ی کل آیین‌نامه‌ها» | Map-Reduce، GraphRAG |
| ۵ | استدلال چندمرحله‌ای (Multi-hop) | ترکیب چند سند | Multi-Query، Agentic RAG |
| ۶ | داده‌ی لحظه‌ای/ساختاریافته | «مانده‌ی مرخصی من؟» | Live tool / API |
| ۷ | Hallucination ممکنه | عدد اشتباه با لحن قاطع | Grounding prompt، Faithfulness eval |
| ۸ | محدودیت Context Window | K زیاد = نویز + هزینه | Rerank، Compression |
| ۹ | تأخیر و هزینه | پاسخ کند | Caching، مدل کوچک‌تر |
| ۱۰ | ایندکس قدیمی (Staleness) | آیین‌نامه‌ی جدید، پاسخ قدیمی | Incremental Update |
| ۱۱ | ریسک امنیتی | Prompt Injection، نشت دسترسی | Access Control، فیلتر محتوا |
| ۱۲ | ارزیابی سخت | «بهتر شد یا نه؟» | Eval Dataset |

## ۳. کِی RAG مناسبه؟

- پرسش‌وپاسخ روی اسناد داخلی (آیین‌نامه، مستندات)
- پشتیبانی مشتری روی پایگاه دانش
- جستجوی معنایی در قراردادها (نیاز به citation)
- دستیار کد روی مستندات
- Onboarding کارمند جدید

## ۴. کِی RAG مناسب نیست؟

| مورد | جایگزین |
|---|---|
| دانش کوچک و کم‌تغییر (چند ده صفحه) | Long-context prompt |
| داده‌ی جدولی و عددی | Text-to-SQL / Tool Calling |
| تغییر سبک و رفتار مدل | Fine-tuning / Prompt engineering |
| تحلیل تجمیعی روی کل داده | SQL / pandas |
| خلاصه‌ی کل پیکره | Map-Reduce summarization |
| تصمیم پرخطر بدون نظارت | Human-in-the-loop |

## ۵. چارچوب تصمیم‌گیری

```
دانش > context window یا مرتب در حال تغییر؟
 ├─ خیر → Prompt مستقیم
 └─ بله
     ├─ عددی/تجمیعی/ساختاریافته؟ → SQL / Tool
     ├─ تغییر رفتار/سبک؟          → Fine-tuning
     └─ متنی + نیازمند منبع؟       → RAG ✅
```

## ۶. الگوهای production

- **Confidence Gate:** اگه امتیاز بهترین hit زیر آستانه بود، LLM صدا زده نمی‌شه و سؤال به انسان ارجاع داده می‌شه.
- **Query Router:** سؤال شخصی/تجمیعی به Tool یا SQL می‌ره، نه vector search.
- آستانه‌ها باید با **داده‌ی تست واقعی** تنظیم بشن، نه حدسی.

## ۷. ایده‌ی کد این نوت‌بوک

- **نمایش محدودیت ۲:** chunking ثابت شرط سند رو از جمله‌ی اصلی جدا می‌کنه.
- **نمایش محدودیت ۳:** شمارش با top-k اشتباهه، شمارش با «tool» درسته.
- **Confidence Gate:** سؤال بی‌ربط بدون فراخوانی LLM به انسان ارجاع می‌شه.
- **تصمیم‌یار ساده:** تابعی که با چند ویژگی سناریو، رویکرد مناسب (Prompt / RAG / SQL / Fine-tuning) رو پیشنهاد می‌ده.

## ۸. چک‌لیست production

- [ ] قبل از انتخاب RAG، جایگزین‌های ساده‌تر (Prompt، SQL، Tool) بررسی شده
- [ ] سؤال‌های شخصی و تجمیعی از مسیر RAG جدا شدن (Router)
- [ ] Confidence Gate با آستانه‌ی تنظیم‌شده روی داده‌ی تست وجود داره
- [ ] حالت «نمی‌دونم» و ارجاع به انسان پیاده‌سازی شده
- [ ] ایندکس مسیر به‌روزرسانی داره (Staleness)
- [ ] کنترل دسترسی و دفاع در برابر Prompt Injection در نظر گرفته شده
- [ ] کیفیت Retrieval و Generation جداگانه قابل اندازه‌گیریه

In [1]:
import re
import zlib
from dataclasses import dataclass, field

import numpy as np


# ======================================================================
# 0) ابزارهای پایه (Embedder و Store ساده)
# ======================================================================
class ToyEmbedder:
    STOPWORDS = {"و", "در", "به", "از", "که", "را", "با", "این", "است", "هر", "می", "ده", "چی", "چند"}

    def __init__(self, dim=256):
        self.dim = dim

    def _embed(self, text):
        vec = np.zeros(self.dim, dtype=np.float32)
        for tok in re.findall(r"\w+", text.lower()):
            if tok not in self.STOPWORDS:
                vec[zlib.crc32(tok.encode()) % self.dim] += 1.0
        n = np.linalg.norm(vec)
        return vec / n if n > 0 else vec

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


@dataclass
class Chunk:
    id: str
    text: str
    metadata: dict = field(default_factory=dict)
    score: float = 0.0


class InMemoryStore:
    def __init__(self):
        self.chunks, self.matrix = [], None

    def upsert(self, chunks, vectors):
        self.chunks += chunks
        new = np.vstack(vectors)
        self.matrix = new if self.matrix is None else np.vstack([self.matrix, new])

    def search(self, q_vec, k=5, filters=None):
        filters = filters or {}
        scores = self.matrix @ q_vec
        hits = []
        for i in np.argsort(-scores):
            if scores[i] <= 0:
                break
            c = self.chunks[i]
            if any(c.metadata.get(key) not in allowed for key, allowed in filters.items()):
                continue
            hits.append(Chunk(c.id, c.text, c.metadata, float(scores[i])))
            if len(hits) == k:
                break
        return hits


def build_store(texts, embedder, metadata=None):
    store = InMemoryStore()
    chunks = [
        Chunk(id=f"c{i}", text=t, metadata=(metadata[i] if metadata else {}))
        for i, t in enumerate(texts)
    ]
    store.upsert(chunks, embedder.embed_documents(texts))
    return store


embedder = ToyEmbedder()


# ======================================================================
# 1) محدودیت ۲: chunking ثابت، شرط رو از جمله‌ی اصلی جدا می‌کنه
# ======================================================================
print("=" * 60, "\n۱) از دست رفتن context در chunking\n" + "=" * 60)

doc = ("مرخصی استحقاقی کارمندان ۳۰ روز در سال است. "
       "اما فقط کارکنان دارای بیش از یک سال سابقه می‌توانند استفاده کنند.")

def fixed_chunks(text, size_words=8):
    words = text.split()
    return [" ".join(words[i:i + size_words]) for i in range(0, len(words), size_words)]

chunks = fixed_chunks(doc, size_words=8)
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {c}")

store1 = build_store(chunks, embedder)
top = store1.search(embedder.embed_query("مرخصی استحقاقی چند روز است"), k=1)[0]
has_condition = "یک سال" in top.text
print(f"\n  chunk برگشتی: {top.text}")
print(f"  شامل شرط «یک سال سابقه» است؟ {'بله ✅' if has_condition else 'خیر ❌ (جواب ناقص و گمراه‌کننده)'}")
print("  راه‌حل‌ها: chunk overlap، sentence-based chunking، parent-child (فاز ۳ و ۷)")


# ======================================================================
# 2) محدودیت ۳: سؤال تجمیعی؛ top-k نمی‌تونه بشماره
# ======================================================================
print("\n" + "=" * 60, "\n۲) سؤال تجمیعی (Aggregation)\n" + "=" * 60)

records = [
    {"emp": f"E{i}", "type": "بدون حقوق", "year": 1403}
    for i in range(1, 7)
] + [{"emp": "E7", "type": "استحقاقی", "year": 1403}]

record_texts = [f"درخواست مرخصی {r['type']} کارمند {r['emp']} ثبت شد" for r in records]
store2 = build_store(record_texts, embedder)

question = "چند نفر مرخصی بدون حقوق گرفتند؟"
rag_hits = store2.search(embedder.embed_query(question), k=3)
rag_count = len(rag_hits)

tool_count = sum(1 for r in records if r["type"] == "بدون حقوق")   # معادل SQL: COUNT(*)

print(f"  سؤال: {question}")
print(f"  شمارش با RAG (top_k=3) : {rag_count}  ← فقط همان K تکه دیده شد")
print(f"  شمارش با Tool/SQL      : {tool_count}  ← درست")
print("  نتیجه: سؤال‌های «چند تا / میانگین / مجموع» باید به SQL یا Tool بروند.")


# ======================================================================
# 3) Confidence Gate
# ======================================================================
print("\n" + "=" * 60, "\n۳) Confidence Gate\n" + "=" * 60)

MIN_SCORE = 0.30

kb = [
    "مرخصی استحقاقی هر کارمند ۳۰ روز در سال است.",
    "بیمه تکمیلی هزینه‌های درمان و دندانپزشکی را پوشش می‌دهد.",
]
store3 = build_store(kb, embedder)

def answer_with_gate(question, store, embedder, min_score=MIN_SCORE):
    hits = store.search(embedder.embed_query(question), k=3)
    best = hits[0].score if hits else 0.0
    if best < min_score:
        return {
            "answer": "اطلاعات کافی در اسناد پیدا نشد. سؤال به کارشناس منابع انسانی ارجاع داده شد.",
            "escalated": True,
            "best_score": round(best, 2),
        }
    return {
        "answer": f"(LLM صدا زده می‌شود) context: {hits[0].text}",
        "escalated": False,
        "best_score": round(best, 2),
    }

for q in ["مرخصی استحقاقی چند روزه؟", "بیمه تکمیلی چی پوشش می‌ده؟", "پایتخت فرانسه کجاست؟"]:
    r = answer_with_gate(q, store3, embedder)
    flag = "🚨 ارجاع به انسان" if r["escalated"] else "✅ جواب با LLM"
    print(f"  [{flag}] score={r['best_score']} | {q}")


# ======================================================================
# 4) تصمیم‌یار ساده: Prompt / RAG / SQL / Fine-tuning
# ======================================================================
print("\n" + "=" * 60, "\n۴) تصمیم‌یار انتخاب رویکرد\n" + "=" * 60)

def choose_approach(corpus_tokens, changes_often, structured_or_aggregate,
                    needs_citation, goal_is_style_change, context_window=100_000):
    if goal_is_style_change:
        return "Fine-tuning / Prompt engineering"
    if structured_or_aggregate:
        return "SQL / Tool Calling"
    if corpus_tokens < context_window * 0.5 and not changes_often:
        return "Prompt مستقیم (long-context)" + (" + ارجاع دستی" if needs_citation else "")
    return "RAG"

scenarios = {
    "آیین‌نامه‌ی ۱۵ صفحه‌ای ثابت":              dict(corpus_tokens=8_000, changes_often=False, structured_or_aggregate=False, needs_citation=True,  goal_is_style_change=False),
    "۵٬۰۰۰ صفحه آیین‌نامه و FAQ در حال تغییر":   dict(corpus_tokens=3_000_000, changes_often=True,  structured_or_aggregate=False, needs_citation=True,  goal_is_style_change=False),
    "مانده‌ی مرخصی و گزارش تجمیعی":              dict(corpus_tokens=0, changes_often=True, structured_or_aggregate=True,  needs_citation=False, goal_is_style_change=False),
    "پاسخ‌دهی با لحن رسمی شرکت":                 dict(corpus_tokens=0, changes_often=False, structured_or_aggregate=False, needs_citation=False, goal_is_style_change=True),
}
for name, params in scenarios.items():
    print(f"  • {name}\n      → {choose_approach(**params)}")

۱) از دست رفتن context در chunking
  chunk 0: مرخصی استحقاقی کارمندان ۳۰ روز در سال است.
  chunk 1: اما فقط کارکنان دارای بیش از یک سال
  chunk 2: سابقه می‌توانند استفاده کنند.

  chunk برگشتی: مرخصی استحقاقی کارمندان ۳۰ روز در سال است.
  شامل شرط «یک سال سابقه» است؟ خیر ❌ (جواب ناقص و گمراه‌کننده)
  راه‌حل‌ها: chunk overlap، sentence-based chunking، parent-child (فاز ۳ و ۷)

۲) سؤال تجمیعی (Aggregation)
  سؤال: چند نفر مرخصی بدون حقوق گرفتند؟
  شمارش با RAG (top_k=3) : 3  ← فقط همان K تکه دیده شد
  شمارش با Tool/SQL      : 6  ← درست
  نتیجه: سؤال‌های «چند تا / میانگین / مجموع» باید به SQL یا Tool بروند.

۳) Confidence Gate
  [✅ جواب با LLM] score=0.41 | مرخصی استحقاقی چند روزه؟
  [✅ جواب با LLM] score=0.61 | بیمه تکمیلی چی پوشش می‌ده؟
  [🚨 ارجاع به انسان] score=0.0 | پایتخت فرانسه کجاست؟

۴) تصمیم‌یار انتخاب رویکرد
  • آیین‌نامه‌ی ۱۵ صفحه‌ای ثابت
      → Prompt مستقیم (long-context) + ارجاع دستی
  • ۵٬۰۰۰ صفحه آیین‌نامه و FAQ در حال تغییر
      → RAG
  • مانده‌ی مرخصی و گزارش تجمیعی
 